# 04 ESDA Exploration

            This notebook runs and interprets spatial autocorrelation: Global Moran's I, Local Moran's I/LISA,
            and Getis-Ord Gi* hot spot analysis.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd))
elif (cwd / "notebooks" / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd / "notebooks"))
else:
    raise FileNotFoundError("Could not find notebook_helpers.py. Run this notebook from the project root or notebooks folder.")

from notebook_helpers import (
    PROJECT_ROOT,
    add_project_root_to_path,
    command_string,
    find_files,
    load_yaml_config,
    notebook_metadata,
    path_status,
    plot_raster,
    print_path_status,
    project_path,
    raster_info,
    raster_stats,
    read_vector,
    run_command,
)

add_project_root_to_path()
RUN_COMMANDS = False  # Change to True only when you want notebook cells to execute CLI scripts.
YEAR = 2023
notebook_metadata("ESDA Exploration", YEAR)

## Step 1 - Check prerequisite surfaces

In [ ]:
import pandas as pd

            prereqs = {
                "LST raster": f"data/processed/lst/lst_ibadan_{YEAR}_celsius.tif",
                "NDVI raster": f"data/processed/indices/ndvi_{YEAR}.tif",
                "NDBI raster": f"data/processed/indices/ndbi_{YEAR}.tif",
                "UHI intensity": f"data/processed/uhi/uhi_intensity_{YEAR}.tif",
            }
            pd.DataFrame(path_status(prereqs))

## Step 2 - Compute UHI intensity if needed

In [ ]:
run_command(["python", "scripts/05_compute_uhi_intensity.py", "--year", YEAR, "--reference", "auto"], dry_run=not RUN_COMMANDS)

## Step 3 - Run ESDA for LST

In [ ]:
run_command(["python", "scripts/06_run_esda.py", "--target", "lst", "--year", YEAR], dry_run=not RUN_COMMANDS)

## Step 4 - Read Global Moran's I result

In [ ]:
import pandas as pd

            moran_path = project_path(f"data/processed/esda/morans_i_{YEAR}.csv")
            if moran_path.exists():
                display(pd.read_csv(moran_path))
            else:
                print("Moran's I table not found yet.")

## Step 5 - Map LISA clusters

In [ ]:
lisa_path = project_path(f"data/processed/esda/lisa_clusters_{YEAR}.gpkg")
            if lisa_path.exists():
                lisa = read_vector(lisa_path)
                display(lisa[["grid_id", "lisa_i", "lisa_p", "lisa_cluster"]].head())
                ax = lisa.plot(column="lisa_cluster", categorical=True, legend=True, figsize=(9, 8), edgecolor="none")
                ax.set_title(f"LISA clusters {YEAR}")
                ax.set_axis_off()
            else:
                print("LISA output not found yet.")

## Step 6 - Map Getis-Ord Gi* hot spots

In [ ]:
gi_path = project_path(f"data/processed/esda/gistar_hotspots_{YEAR}.gpkg")
            if gi_path.exists():
                gi = read_vector(gi_path)
                display(gi[["grid_id", "gi_z", "gi_p", "gistar_cluster"]].head())
                ax = gi.plot(column="gistar_cluster", categorical=True, legend=True, figsize=(9, 8), edgecolor="none")
                ax.set_title(f"Getis-Ord Gi* hot spots {YEAR}")
                ax.set_axis_off()
            else:
                print("Gi* output not found yet.")